# ATML PA2 - Task 2 evaluations on a second Colab account

Thin launcher only. Runs the `midpoint` and `standard` held-out evaluations of Task 2 on an account whose Drive did not train them.

Before starting:

- Set the runtime to a T4 GPU (Runtime > Change runtime type), the same GPU as every other Task 2 run.
- Upload `adapter_config.json` and `adapter_model.safetensors` of the standard run to `MyDrive/atml_pa2/outputs/task2_ppo/standard/` on this account's Drive.

### 1. GPU check

Must show `Tesla T4`.

In [ ]:
!nvidia-smi

### 2. Clone or update the repo

In [ ]:
%cd /content
!test -d atml_PA2 || git clone https://github.com/Zeeba1705/atml_PA2.git
%cd /content/atml_PA2
!git fetch origin
!git checkout task2-ppo
!git pull
!git log --oneline -3

### 3. Install the pinned requirements

In [ ]:
!pip install -q -r requirements.txt

### 4. Mount Drive and link the large folders

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

In [ ]:
%%bash
DRIVE=/content/drive/MyDrive/atml_pa2

for d in checkpoints cached outputs results; do
    mkdir -p "$DRIVE/$d"

    if [ ! -L "$d" ]; then
        if [ -d "$d" ]; then
            cp -rn "$d/." "$DRIVE/$d/"
            rm -rf "$d"
        fi

        ln -s "$DRIVE/$d" "$d"
    fi
done

ls -l

### 5. Course assets

Downloads only when validation fails.

In [ ]:
!python -m scripts.validate_assets || (python -m scripts.download_assets && python -m scripts.validate_assets)

### 6. Check the copied standard adapter

Expect `adapter_config.json` and `adapter_model.safetensors` (about 4.4 MB). If the folder is empty, Drive has not synced the upload yet: wait a minute and rerun.

In [ ]:
!ls -la outputs/task2_ppo/standard

### 7. Midpoint evaluation

200 held-out prompts, about 40 minutes. An interrupted evaluation starts over, so keep the tab active.

In [ ]:
!python -m task2_ppo.evaluate --config configs/ppo.yaml --adapter checkpoints/ppo_midpoint_policy --name midpoint

### 8. Standard evaluation

200 held-out prompts, about 40 minutes.

In [ ]:
!python -m task2_ppo.evaluate --config configs/ppo.yaml --adapter outputs/task2_ppo/standard --name standard

### 9. Confirm both were saved

In [ ]:
!ls -la results/task2_ppo/midpoint results/task2_ppo/standard
!python -c "import json; [print(n, {k: round(json.load(open(f'results/task2_ppo/{n}/eval_metrics.json'))['heldout'][k], 4) for k in ['reward_mean','kl_token_mean','length_mean','no_eos_fraction']}) for n in ['midpoint','standard']]"